# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# ML-07 — Baseline Action Score and Top-20 Review

## 1. My rule and its reason codes

I use a simple baseline to rank content pages for possible refresh review.

The rule gives a higher score to pages that have **meaningful search visibility but weaker-than-expected CTR for their search-position band**. GSC impressions are used as a second signal so that pages with more observed search visibility receive more attention.

This is a **directional decision-support score**, not a prediction of refresh success. It does not use future outcomes, product flags, URLs, client names, or private queries.

### Reason codes

* **LOW_CTR_HIGH_VISIBILITY** — observed CTR is below the position-band benchmark while the page has meaningful impressions.
* **LOW_CTR_MODERATE_VISIBILITY** — observed CTR is below the position-band benchmark with moderate impressions.
* **POSITION_BAND_OPPORTUNITY** — the page has search visibility in a position band where improving CTR may be useful.
* **NO_CLEAR_OPPORTUNITY** — the observed signals do not provide a strong refresh indication.

The final ranking is used to prioritize a small review queue. The score should not be interpreted as proof that a refresh will improve performance.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# ML-07 — Build baseline ranked queue

import pandas as pd
import numpy as np
from pathlib import Path

# Use the same streamed warehouse source inspected in ML-04.
rows = list(ds.take(1000))
df = pd.DataFrame(rows)

# Convert required fields to numeric values.
df["gsc_impressions"] = pd.to_numeric(
    df["gsc_impressions"], errors="coerce"
)

df["gsc_clicks"] = pd.to_numeric(
    df["gsc_clicks"], errors="coerce"
)

df["gsc_avg_position"] = pd.to_numeric(
    df["gsc_avg_position"], errors="coerce"
)

# Remove rows that cannot support the baseline calculation.
df = df.dropna(
    subset=[
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
    ]
).copy()

# Calculate observed CTR.
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Create broad search-position bands.
def position_band(position):
    if position <= 3:
        return "1-3"
    elif position <= 5:
        return "4-5"
    elif position <= 10:
        return "6-10"
    elif position <= 20:
        return "11-20"
    else:
        return "21+"

df["position_band"] = df["gsc_avg_position"].apply(position_band)

# Calculate the median CTR within each observed position band.
band_benchmark = (
    df.groupby("position_band")["ctr"]
    .median()
    .rename("band_median_ctr")
)

df = df.merge(
    band_benchmark,
    on="position_band",
    how="left"
)

# CTR gap relative to the position-band benchmark.
df["ctr_gap"] = (
    df["band_median_ctr"] - df["ctr"]
).clip(lower=0)

# Normalize impressions using log1p so very large values
# do not completely dominate the score.
df["impression_signal"] = np.log1p(
    df["gsc_impressions"]
)

max_impressions = df["impression_signal"].max()

if max_impressions > 0:
    df["impression_signal_norm"] = (
        df["impression_signal"] / max_impressions
    )
else:
    df["impression_signal_norm"] = 0

# Normalize CTR opportunity.
max_ctr_gap = df["ctr_gap"].max()

if max_ctr_gap > 0:
    df["ctr_opportunity"] = (
        df["ctr_gap"] / max_ctr_gap
    )
else:
    df["ctr_opportunity"] = 0

# Baseline score:
# 70% CTR opportunity + 30% observed search visibility.
df["baseline_action_score"] = (
    0.70 * df["ctr_opportunity"]
    + 0.30 * df["impression_signal_norm"]
)

# Assign transparent reason codes.
def reason_code(row):
    if row["ctr"] < row["band_median_ctr"]:
        if row["gsc_impressions"] >= df["gsc_impressions"].median():
            return "LOW_CTR_HIGH_VISIBILITY"
        else:
            return "LOW_CTR_MODERATE_VISIBILITY"
    elif row["gsc_impressions"] >= df["gsc_impressions"].median():
        return "POSITION_BAND_OPPORTUNITY"
    else:
        return "NO_CLEAR_OPPORTUNITY"

df["reason_code"] = df.apply(reason_code, axis=1)

# Rank all observations.
df = df.sort_values(
    by=[
        "baseline_action_score",
        "gsc_impressions",
        "ctr"
    ],
    ascending=[
        False,
        False,
        True
    ]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Create output directory.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Keep the output focused on decision-support fields.
output_cols = [
    "rank",
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "position_band",
    "band_median_ctr",
    "ctr_gap",
    "baseline_action_score",
    "reason_code",
]

ranked_queue = df[output_cols].copy()

output_path = output_dir / "baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)

print("Rows ranked:", len(ranked_queue))
print("Output written to:", output_path)
print("\nTop 10:")
display(ranked_queue.head(10))

Rows ranked: 1000
Output written to: work/outputs/baseline_action_score.csv

Top 10:


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,position_band,band_median_ctr,ctr_gap,baseline_action_score,reason_code
0,1,2025-01-28,client_9958f0a7ae1df715,content_f94fe855380e150f,303,3,1.811881,0.009901,1-3,0.0,0.0,0.300000,POSITION_BAND_OPPORTUNITY
1,2,2025-01-27,client_9958f0a7ae1df715,content_f94fe855380e150f,266,6,1.812030,0.022556,1-3,0.0,0.0,0.293190,POSITION_BAND_OPPORTUNITY
2,3,2025-01-29,client_9958f0a7ae1df715,content_f94fe855380e150f,240,5,1.808333,0.020833,1-3,0.0,0.0,0.287814,POSITION_BAND_OPPORTUNITY
3,4,2025-01-30,client_9958f0a7ae1df715,content_37b3bafd5f88fdd1,202,4,5.034653,0.019802,6-10,0.0,0.0,0.278810,POSITION_BAND_OPPORTUNITY
4,5,2025-01-28,client_9958f0a7ae1df715,content_d02be57d816cf3d7,174,1,3.017241,0.005747,4-5,0.0,0.0,0.271021,POSITION_BAND_OPPORTUNITY
5,6,2025-01-27,client_9958f0a7ae1df715,content_d02be57d816cf3d7,118,0,3.440678,0.000000,4-5,0.0,0.0,0.250784,POSITION_BAND_OPPORTUNITY
6,7,2025-01-29,client_9958f0a7ae1df715,content_d02be57d816cf3d7,97,0,4.051546,0.000000,4-5,0.0,0.0,0.240595,POSITION_BAND_OPPORTUNITY
7,8,2025-01-30,client_9958f0a7ae1df715,content_de7b08874af74c00,87,0,7.160920,0.000000,6-10,0.0,0.0,0.234947,POSITION_BAND_OPPORTUNITY
8,9,2025-01-29,client_9958f0a7ae1df715,content_84f5a9ecfefa108e,81,1,2.493827,0.012346,1-3,0.0,0.0,0.231242,POSITION_BAND_OPPORTUNITY
9,10,2025-01-27,client_9958f0a7ae1df715,content_de7b08874af74c00,80,1,6.012500,0.012500,6-10,0.0,0.0,0.230598,POSITION_BAND_OPPORTUNITY


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

The top-20 queue is generated directly from the baseline score.

Each recommendation is treated as a **review candidate**, not as a guaranteed refresh decision.

For each candidate, the review records:

* **Action** — whether the page should be reviewed for a possible refresh.
* **Reason code** — the observed signal that caused the page to rank highly.
* **Confidence note** — how strong the observed evidence is.
* **What would make it wrong** — an explicit limitation or alternative explanation.

Confidence is intentionally qualitative because this is a transparent baseline rather than a validated predictive model.

A high score means the observed GSC signals look more promising for review. It does **not** establish that refreshing the page will improve future performance.


In [9]:
# ML-07 — Top-20 review

top20 = ranked_queue.head(20).copy()

def action_for_row(row):
    if row["reason_code"] in [
        "LOW_CTR_HIGH_VISIBILITY",
        "LOW_CTR_MODERATE_VISIBILITY"
    ]:
        return "Review for refresh"
    elif row["reason_code"] == "POSITION_BAND_OPPORTUNITY":
        return "Review if resources allow"
    else:
        return "Low-priority review"

def confidence_for_row(row):
    if row["gsc_impressions"] >= ranked_queue["gsc_impressions"].quantile(0.75):
        return "Moderate: higher observed visibility"
    elif row["gsc_impressions"] >= ranked_queue["gsc_impressions"].median():
        return "Directional: moderate observed visibility"
    else:
        return "Low: limited observed visibility"

def wrong_if(row):
    return (
        "The ranking could be wrong if the position/CTR observation "
        "is noisy, the page has limited impressions, or the observed "
        "CTR gap does not translate into a future refresh opportunity."
    )

top20["action"] = top20.apply(action_for_row, axis=1)
top20["confidence_note"] = top20.apply(confidence_for_row, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review_cols = [
    "rank",
    "content_hash_id",
    "report_date",
    "baseline_action_score",
    "gsc_impressions",
    "gsc_avg_position",
    "ctr",
    "position_band",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong",
]

top20_review = top20[review_cols].copy()

display(top20_review)

,rank,content_hash_id,report_date,baseline_action_score,gsc_impressions,gsc_avg_position,ctr,position_band,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_f94fe855380e150f,2025-01-28,0.300000,303,1.811881,0.009901,1-3,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
1,2,content_f94fe855380e150f,2025-01-27,0.293190,266,1.812030,0.022556,1-3,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
2,3,content_f94fe855380e150f,2025-01-29,0.287814,240,1.808333,0.020833,1-3,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
3,4,content_37b3bafd5f88fdd1,2025-01-30,0.278810,202,5.034653,0.019802,6-10,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
4,5,content_d02be57d816cf3d7,2025-01-28,0.271021,174,3.017241,0.005747,4-5,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
5,6,content_d02be57d816cf3d7,2025-01-27,0.250784,118,3.440678,0.000000,4-5,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
6,7,content_d02be57d816cf3d7,2025-01-29,0.240595,97,4.051546,0.000000,4-5,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
7,8,content_de7b08874af74c00,2025-01-30,0.234947,87,7.160920,0.000000,6-10,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
8,9,content_84f5a9ecfefa108e,2025-01-29,0.231242,81,2.493827,0.012346,1-3,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...
9,10,content_de7b08874af74c00,2025-01-27,0.230598,80,6.012500,0.012500,6-10,POSITION_BAND_OPPORTUNITY,Review if resources allow,Moderate: higher observed visibility,The ranking could be wrong if the position/CTR...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

Some high-ranked observations may still be weak recommendations.

A weak pick can occur when a page has relatively low impressions, when the position-band benchmark is unstable because of limited observations, or when a CTR difference is small and may be ordinary variation.

These recommendations should therefore be treated as **review candidates**, not confirmed refresh priorities.

The leakage check verifies that the baseline uses only information available in the current observation. The score uses GSC impressions, clicks, average position, observed CTR, and position-band comparisons. It does not use a future performance window, refresh outcome, product flag, client name, URL, or private search query.

Because the inspected warehouse sample contains daily observations, this baseline should also not be interpreted as a causal estimate of refresh impact.


In [10]:
# ML-07 — Weak-pick and leakage checks

print("WEAK-PICK CHECK")
print("=" * 50)

# Identify lower-visibility items among the top 20.
visibility_cutoff = ranked_queue["gsc_impressions"].quantile(0.25)

weak_picks = top20[
    top20["gsc_impressions"] <= visibility_cutoff
].copy()

print("Top-20 rows:", len(top20))
print("Lower-visibility top-20 rows:", len(weak_picks))

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "rank",
                "content_hash_id",
                "gsc_impressions",
                "gsc_avg_position",
                "ctr",
                "baseline_action_score",
                "reason_code",
            ]
        ]
    )
else:
    print("No top-20 rows fell into the lower-visibility quartile.")


print("\nLEAKAGE CHECK")
print("=" * 50)

# Fields actually used to construct the score.
score_input_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "position_band",
    "band_median_ctr",
    "ctr_gap",
]

print("Score input fields:")
for field in score_input_fields:
    print("-", field)

# Explicitly check for suspicious future/product/query fields.
forbidden_terms = [
    "future",
    "outcome",
    "refresh",
    "product",
    "query",
    "url",
]

suspicious_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("\nPotentially sensitive/future-looking columns present in working data:")
print(suspicious_columns)

# Verify that score columns are finite.
score_is_finite = np.isfinite(
    ranked_queue["baseline_action_score"]
).all()

print("\nAll baseline scores finite:", score_is_finite)

# Verify ranking is descending.
ranking_is_valid = (
    ranked_queue["baseline_action_score"]
    .is_monotonic_decreasing
)

print("Scores ranked descending:", ranking_is_valid)

print("\nLeakage conclusion:")
print(
    "The baseline score was constructed from observed GSC performance "
    "signals only. No future-window outcome or refresh-success label "
    "was used."
)

WEAK-PICK CHECK
Top-20 rows: 20
Lower-visibility top-20 rows: 0
No top-20 rows fell into the lower-visibility quartile.

LEAKAGE CHECK
Score input fields:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr
- position_band
- band_median_ctr
- ctr_gap

Potentially sensitive/future-looking columns present in working data:
[]

All baseline scores finite: True
Scores ranked descending: True

Leakage conclusion:
The baseline score was constructed from observed GSC performance signals only. No future-window outcome or refresh-success label was used.


## Self-check

* [x] Every section above is filled — markdown thinking and supporting code
* [ ] The notebook runs top to bottom with no errors after Runtime → Run all
* [x] No client names, URLs, or private queries are used as model inputs
* [x] Claims use careful language such as observed, measured, directional, and decision-support
* [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`
* [ ] The completed notebook is committed under `work/notebooks/`
* [ ] The repository URL is ready for submission

### Final interpretation

This baseline is a transparent ranking rule for prioritizing content pages for human review. It identifies pages with relatively weak observed CTR for their search-position band and combines that signal with observed GSC impressions.

The ranking is **directional decision-support**, not evidence that a refresh will definitely improve performance. Future validation would be required before treating the score as a predictive model.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.